# Kandinsky Game→Audio — Step 5: Checkpoint and Pipeline Integration Audit

This notebook reads the pinned Kandinsky source to map where a game-event condition must enter the real system.

It is a **read-only source audit**. It does not patch Kandinsky, download model weights, construct the full model, train, or generate audio.

## Questions answered

- How does the official factory construct the DiT?
- Does it pass a game-token dimension?
- How are checkpoint keys loaded, and what happens when keys are missing?
- Which public pipeline and sampler calls would need a game-token argument?
- What is the smallest safe integration path after this audit?

## 1. Fetch the pinned public source

In [ ]:
import json
import subprocess
from pathlib import Path

K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-step5-source-audit")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True,
    )

assert subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip() == K6_COMMIT

def read_source(relative_path):
    return (K6_DIR / relative_path).read_text(encoding="utf-8")

print("Pinned commit:", K6_COMMIT)
print("Source checkout:", K6_DIR)

## 2. Inspect model construction and checkpoint loading

The factory constructs the model on the meta device, builds constructor arguments from the model config, then calls the official checkpoint loader. We inspect the relevant source without downloading the checkpoint.

In [ ]:
factory = read_source("kandinsky/pipeline/factory.py")
weights = read_source("kandinsky/runtime/weights.py")
lite_config = read_source("kandinsky/configs/modules/transformer_lite.json")

factory_has_game_dim = "game_dim" in factory
factory_constructs_on_meta = 'with torch.device("meta")' in factory
loader_is_strict_assign = "load_state_dict(state_dict, strict=True, assign=True)" in weights
config_has_game_dim = '"game_dim"' in lite_config
materializer_handles_buffers = "for name, buf in list(mod._buffers.items())" in weights
materializer_handles_parameters = "list(mod._parameters.items())" in weights

print(json.dumps({
    "factory_passes_game_dim": factory_has_game_dim,
    "factory_constructs_on_meta_device": factory_constructs_on_meta,
    "lite_model_config_declares_game_dim": config_has_game_dim,
    "checkpoint_loader_uses_strict_true_and_assign": loader_is_strict_assign,
    "meta_materializer_handles_nonpersistent_buffers": materializer_handles_buffers,
    "meta_materializer_handles_missing_parameters": materializer_handles_parameters,
    "pretrained_checkpoint_downloaded": False,
}, indent=2))
assert not factory_has_game_dim
assert factory_constructs_on_meta
assert not config_has_game_dim
assert loader_is_strict_assign
assert materializer_handles_buffers
assert not materializer_handles_parameters

### Relevant factory section

This shows the model constructor arguments and the subsequent checkpoint load.

In [ ]:
def print_excerpt(text, start_marker=None, end_marker=None, label="source excerpt"):
    # Defaults keep the helper safe if an older Colab copy omits the label argument.
    if start_marker is None and end_marker == "checkpoint loading":
        start_marker = "def load_bf16_checkpoint("
        end_marker = None
        if label == "source excerpt":
            label = "checkpoint loading"
    if start_marker is None:
        start_marker = ""
    start = text.find(start_marker)
    if start < 0:
        raise RuntimeError(f"Could not find {label} start marker.")
    end = len(text) if end_marker is None else text.find(end_marker, start + len(start_marker))
    if end < 0:
        raise RuntimeError(f"Could not find {label} end marker.")
    excerpt = text[start:end]
    print(f"--- {label} ---")
    print(excerpt)

print_excerpt(
    factory,
    "def create_bare_dit(",
    "def _build_pipeline(",
    "factory model construction",
)
print_excerpt(
    weights,
    "def load_bf16_checkpoint(",
    None,
    "checkpoint loading",
)

## 3. Inspect public request and denoising call boundaries

The pipeline request currently has no game-event argument. A game condition would need to travel from the request into both supported denoising paths:
- the standard denoise loop, which may call the model twice for text guidance;
- the PiFlow loop, which calls the model on each integration step.

This audit prints those interfaces only; it does not call them.

In [ ]:
pipeline = read_source("kandinsky/pipeline/pipeline.py")
denoise = read_source("kandinsky/core/algo/denoise_loop.py")
piflow = read_source("kandinsky/core/algo/piflow_sampler.py")

def has_line_containing(text, needle):
    return any(needle in line for line in text.splitlines())

call_signature = pipeline.split("def __call__(", 1)[-1].split(") ->", 1)[0]
interface_report = {
    "pipeline_call_has_game_tokens": "game_tokens" in call_signature,
    "standard_sampler_has_game_tokens": "game_tokens" in denoise,
    "piflow_sampler_has_game_tokens": "game_tokens" in piflow,
    "standard_sampler_passes_visual_token_type_ids": "visual_token_type_ids=visual_token_type_ids" in denoise,
    "piflow_sampler_passes_visual_token_type_ids": "visual_token_type_ids=visual_token_type_ids" in piflow,
}
print(json.dumps(interface_report, indent=2))
assert not interface_report["pipeline_call_has_game_tokens"]
assert not interface_report["standard_sampler_has_game_tokens"]
assert not interface_report["piflow_sampler_has_game_tokens"]

print_excerpt(
    pipeline,
    "    def __call__(",
    "        batch_size = ",
    "pipeline public request signature",
)
print_excerpt(
    denoise,
    "        def _forward(te, pe, rope, attn_mask):",
    "        vel_cond = _forward(",
    "standard denoise model call",
)
print_excerpt(
    piflow,
    "        pred_video, pred_audio = dit(",
    "        pred_video = _normalize_grid_prediction",
    "PiFlow model call",
)

## 4. What the source audit means

### Checkpoint loading and the meta-device caveat

The official factory constructs the DiT inside a meta-device context. Its checkpoint loader then uses `load_state_dict(..., strict=True, assign=True)`. An old checkpoint has no keys for new Game→Audio parameters, so strict loading rejects it.

Changing this to plain `strict=False` is also unsafe here: unmatched parameters created on the meta device may remain unmaterialized. The source's `materialize_meta_buffers` helper only allocates non-persistent **buffers**, not missing parameters.

A promising safe prototype is therefore:
1. Construct and load the unmodified base DiT with the existing strict loader.
2. Attach newly initialized Game→Audio modules to its fused blocks after the base weights are loaded.
3. Load a separate adapter state dict containing only those new modules, with its own strict key check.

This keeps the published base checkpoint unchanged. Step 5 only identifies this path from source; Step 6 should prove it with a tiny model before any pipeline edits.

### Pipeline integration

The game condition has to be passed through the full chain:

`Pipeline request → standard / PiFlow sampler → DiT.forward → fused audio blocks`

For standard text-guided sampling, the same game condition should be sent to both conditional and unconditional model calls. The game branch can coexist with text and video; it does not remove the prompt by itself.

### Recommended next step

Build a tiny post-load adapter prototype: strict-load a base model, attach the new modules afterward, save/load only the adapter keys, and confirm all original model parameters remain unchanged. Then wire one sampler path in isolation.

## Source links

All audited code is pinned to Kandinsky 6 commit 7ceeefad7fb625011f7db602c43893a989f19c15:

- [Factory](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/pipeline/factory.py)
- [Checkpoint loader](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/weights.py)
- [Pipeline request](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/pipeline/pipeline.py)
- [Standard denoise loop](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/denoise_loop.py)
- [PiFlow sampler](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/piflow_sampler.py)
- [Lite DiT config](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/configs/modules/transformer_lite.json)